In [20]:
import os
import glob
import pandas as pd

base_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports"

# Adjust the pattern if filenames have a specific pattern, e.g. "*_daily.csv"
csv_files = glob.glob(os.path.join(base_dir, "*.csv"))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_file"] = os.path.basename(f)  # optional, to track origin
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner,FileDate,source_file,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,20250616,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2353388,teletype.in,0,0.0,4.0,16.55%,7-Day: Low confidence,68.97%,14-Day: Possibly active,91.78%,30-Day: Highly likely,VA,20260311,full_daily_report_20260311.csv,75.22%,45-Day: Highly likely,0.0,2.54%,1-Day: Low confidence
2353389,tester.com,0,0.0,0.0,0.03%,7-Day: Low confidence,0.02%,14-Day: Low confidence,0.51%,30-Day: Low confidence,VA,20260311,full_daily_report_20260311.csv,15.16%,45-Day: Low confidence,0.0,0.02%,1-Day: Low confidence
2353390,www.citiquartz.com,0,0.0,1.0,5.2%,7-Day: Low confidence,11.2%,14-Day: Low confidence,65.95%,30-Day: Possibly active,VA,20260311,full_daily_report_20260311.csv,50.09%,45-Day: Possibly active,0.0,0.68%,1-Day: Low confidence
2353391,www.deepseek.com,0,2.0,17.0,97.22%,7-Day: Highly likely,99.72%,14-Day: Highly likely,99.98%,30-Day: Highly likely,VA,20260311,full_daily_report_20260311.csv,98.22%,45-Day: Highly likely,0.0,35.12%,1-Day: Low confidence


In [22]:
# Clean, convert, and aggregate in one step

# Rename 45-day probability column if present
if "ensemble_45d" in all_data.columns:
    all_data = all_data.rename(columns={"ensemble_45d": "Probability: 45-Day"})

# Drop confidence columns and other unwanted fields
cols_to_drop = [c for c in all_data.columns if "confidence" in c.lower()]
cols_to_drop += ["source_file", "FileDate", "Partner", "Observed Today"]
# Remove duplicates while preserving order
cols_to_drop = list(dict.fromkeys(cols_to_drop))
all_data = all_data.drop(columns=cols_to_drop, errors="ignore")

# Convert probability columns from strings like "12.3%" to numeric 0–1 floats
prob_cols = [c for c in all_data.columns if "Probability" in c]
for c in prob_cols:
    all_data[c] = pd.to_numeric(
        all_data[c].astype(str).str.rstrip("%"), errors="coerce"
    ) / 100.0

# Group by Indicator and take average of numeric columns
grouped = all_data.groupby("Indicator", as_index=False).mean(numeric_only=True)

# Round frequency columns to nearest whole number
freq_cols = [c for c in grouped.columns if c.startswith("Frequency ")]
for c in freq_cols:
    grouped[c] = grouped[c].round(0).astype("Int64")

# Convert probabilities (0–1) to percentages with 2 decimals
prob_cols_grouped = [c for c in grouped.columns if "Probability" in c]
for c in prob_cols_grouped:
    grouped[c] = (grouped[c] * 100).round(2)    

grouped

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,1,0.12,0.23,0.34,0.37,0,0.00
1,1-you.njalla.no,0,2,0.25,0.41,0.63,0.50,0,0.09
2,1.192.18.4,0,1,0.16,0.29,0.47,0.43,0,0.00
3,1.204.166.3,0,1,0.12,0.24,0.49,0.41,0,0.00
4,1.234.83.26,3,13,0.68,0.76,0.89,0.76,0,0.35
...,...,...,...,...,...,...,...,...,...
5804,ymath.yonsei.ac.kr,0,0,0.00,0.01,0.03,0.13,0,0.00
5805,yotpo-static.com,0,0,0.01,0.02,0.05,0.14,0,0.00
5806,yourpensionmeeting.com,3,12,0.57,0.67,0.79,0.67,0,0.33
5807,zerocap.com,0,0,0.00,0.00,0.00,NaN,<NA>,NaN


In [ ]:
tas_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"

# Read the first sheet (sheet index 0) into a DataFrame
tas_df = pd.read_excel(tas_path, sheet_name=0)
tas_df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,1.234.83.26,2026-02-14,Address,43,3,0.997644,0,0,"CMS, FDA, HRSA, OS",NaN,NaN,180,469,154,low,[2026-03-11] Severity: low. VT score: 3. Top d...
1,101.168.57.163,2026-02-14,Address,2,3,0.999890,0,0,"CMS, FDA, OS, VA",Incident:INC9407577,NaN,180,590,458,medium,[2026-03-11] Severity: medium. VT score: 11. T...
2,101.71.130.99,2026-03-03,Address,139,3,0.992384,0,0,"CMS, DHA, FDA, HHS, HRSA, IHS, NIH, OS",Incident:INC9270850,NaN,360,436,467,medium,[2026-03-11] Severity: medium. VT score: 11. T...
3,103.114.96.246,2026-02-26,Address,41,1,0.997753,1,0,"CMS, DHA, NIH, OS, VA",NaN,Mr Hamza Group,170,224,145,low,[2026-03-11] Severity: low. VT score: 2. Top d...
4,103.120.116.162,2026-03-11,Address,43,3,0.997644,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:INC9385644,NaN,740,870,364,medium,[2026-03-11] Severity: medium. VT score: 6. To...


In [34]:
merged = grouped.merge(tas_df, on="Indicator", how="left")

# Drop rows where all TAS columns are missing
tas_cols = [c for c in merged.columns if c not in grouped.columns]
merged_nonnull = merged.dropna(subset=tas_cols, how="all").copy()

# Fill missing incidents/events and threat actor fields with text

# Columns related to incidents/events
incident_cols = [c for c in merged_nonnull.columns 
                 if "incident" in c.lower() or "event" in c.lower()]

# Columns related to threat actors
threat_actor_cols = [c for c in merged_nonnull.columns 
                     if "threat actor" in c.lower() or "actor" in c.lower()]

merged_nonnull.loc[:, incident_cols] = merged_nonnull[incident_cols].fillna("No known incidents/events")
merged_nonnull.loc[:, threat_actor_cols] = merged_nonnull[threat_actor_cols].fillna("No known threat actors")

merged_nonnull

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day,Last Observed,...,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,HTOC Threat Score,Severity,Explanation
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,1,0.12,0.23,0.34,0.37,0,0.00,2026-01-08,...,1.0,0.0,VA,Incident:30246,No known threat actors,810.0,905.0,177.0,low,Severity: low. Could not pull VT score to calc...
1,1-you.njalla.no,0,2,0.25,0.41,0.63,0.50,0,0.09,2026-02-03,...,0.0,0.0,"NIH, VA",No known incidents/events,Wicked Panda,180.0,416.0,312.0,medium,[2026-03-05] Severity: medium. VT score not av...
4,1.234.83.26,3,13,0.68,0.76,0.89,0.76,0,0.35,2026-02-14,...,0.0,0.0,"CMS, FDA, HRSA, OS",No known incidents/events,No known threat actors,180.0,469.0,154.0,low,[2026-03-11] Severity: low. VT score: 3. Top d...
5,1.4.195.14,0,2,0.24,0.35,0.55,0.51,0,0.06,2026-01-17,...,1.0,0.0,"CMS, VA",No known incidents/events,No known threat actors,180.0,229.0,134.0,low,Severity: low. VT score: 1. Top drivers: TOR a...
12,101.168.57.163,0,2,0.19,0.42,0.83,0.65,0,0.06,2026-02-14,...,0.0,0.0,"CMS, FDA, OS, VA",Incident:INC9407577,No known threat actors,180.0,590.0,458.0,medium,[2026-03-11] Severity: medium. VT score: 11. T...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5762,www.deepseek.com,2,11,0.63,0.73,0.82,0.74,0,0.24,2026-03-10,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",No known incidents/events,No known threat actors,180.0,469.0,50.0,low,[2026-03-11] Severity: low. VT score not avail...
5763,www.deepseek.com.cdn.cloudflare.net,1,4,0.23,0.32,0.44,0.41,0,0.04,2025-11-28,...,0.0,0.0,"CMS, DHA, IHS, NIH, VA",No known incidents/events,No known threat actors,170.0,464.0,293.0,medium,Severity: medium. Could not pull VT score to c...
5792,www.prosinthecity.com,0,1,0.08,0.16,0.33,0.34,0,0.03,2025-11-12,...,0.0,0.0,"CMS, DHA, NIH, OS",Incident:6755399448004157;Incident:546481,No known threat actors,170.0,363.0,417.0,medium,Severity: medium. Could not pull VT score to c...
5795,www.sthda.com,1,8,0.45,0.59,0.70,0.64,0,0.11,2026-03-03,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:331131,No known threat actors,180.0,368.0,153.0,low,[2026-03-11] Severity: low. VT score not avail...


Threat Recurrence Risk Idex

In [ ]:
# Total TRRI scores across all (assumed active) indicators

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

total_trri

In [ ]:
# Step 5 — Create TRRI risk zones per horizon

# Bins and labels based on TRRI value
bins = [0, 100, 300, 600, float("inf")]
labels = [
    "Low Recurrence Risk",        # < 100
    "Moderate Recurrence Risk",   # 100–299
    "High Recurrence Risk",       # 300–599
    "Critical Recurrence Risk",   # 600+
]

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

for col in trri_cols:
    risk_col = col.replace("TRRI_", "Risk_")
    merged_nonnull[risk_col] = pd.cut(
        merged_nonnull[col],
        bins=bins,
        labels=labels,
        right=False,  # include lower bound, exclude upper (so 100 goes to Moderate, 300 to High, etc.)
    )

# Quick look at indicator, TRRI, and risk zones
cols_to_show = ["Indicator"] + trri_cols + [c for c in merged_nonnull.columns if c.startswith("Risk_")]
merged_nonnull[cols_to_show].head()

In [36]:
import pandas as pd

preferred_score_col = "HTOC Threat Score"

if preferred_score_col in merged_nonnull.columns:
    score_col = preferred_score_col
else:
    score_cols = [c for c in merged_nonnull.columns if "threat score" in c.lower()]
    if not score_cols:
        raise ValueError("No Threat Score column found; check TAS column names.")
    score_col = score_cols[0]

merged_nonnull[score_col] = pd.to_numeric(merged_nonnull[score_col], errors="coerce")

prob_map = {
    "1d": "Probability: 1-Day",
    "7d": "Probability: 7-Day",
    "14d": "Probability: 14-Day",
    "30d": "Probability: 30-Day",
    "45d": "Probability: 45-Day",
}

for suffix, pcol in prob_map.items():
    if pcol not in merged_nonnull.columns:
        continue

    merged_nonnull[pcol] = pd.to_numeric(merged_nonnull[pcol], errors="coerce")

    max_prob = merged_nonnull[pcol].max(skipna=True)

    # If max > 1, assume percentages like 12, 25, 68
    if pd.notna(max_prob) and max_prob > 1:
        prob_series = merged_nonnull[pcol] / 100.0
    else:
        prob_series = merged_nonnull[pcol]

    merged_nonnull[f"TRRI_{suffix}"] = (
        merged_nonnull[score_col] * prob_series
    ).round(1)

display_cols = ["Indicator", score_col] + [f"TRRI_{k}" for k in prob_map if f"TRRI_{k}" in merged_nonnull.columns]
merged_nonnull[display_cols]

,Indicator,HTOC Threat Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,177.0,0.0,21.2,40.7,60.2,65.5
1,1-you.njalla.no,312.0,28.1,78.0,127.9,196.6,156.0
4,1.234.83.26,154.0,53.9,104.7,117.0,137.1,117.0
5,1.4.195.14,134.0,8.0,32.2,46.9,73.7,68.3
12,101.168.57.163,458.0,27.5,87.0,192.4,380.1,297.7
...,...,...,...,...,...,...,...
5762,www.deepseek.com,50.0,12.0,31.5,36.5,41.0,37.0
5763,www.deepseek.com.cdn.cloudflare.net,293.0,11.7,67.4,93.8,128.9,120.1
5792,www.prosinthecity.com,417.0,12.5,33.4,66.7,137.6,141.8
5795,www.sthda.com,153.0,16.8,68.9,90.3,107.1,97.9


In [46]:
# Step 5 — Create TRRI risk zones per horizon

# Bins and labels based on TRRI value
bins = [0, 200, 400, 600, float("inf")]
labels = [
    "Low Recurrence Risk",        # < 200
    "Moderate Recurrence Risk",   # 200–400
    "High Recurrence Risk",       # 400–600
    "Critical Recurrence Risk",   # 600+
]

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

for col in trri_cols:
    risk_col = col.replace("TRRI_", "Risk_")
    merged_nonnull[risk_col] = pd.cut(
        merged_nonnull[col],
        bins=bins,
        labels=labels,
        right=False,  # include lower bound, exclude upper (so 100 goes to Moderate, 300 to High, etc.)
    )

# Quick look at indicator, TRRI, and risk zones
cols_to_show = ["Indicator"] + trri_cols + [c for c in merged_nonnull.columns if c.startswith("Risk_")]
merged_nonnull[cols_to_show]

,Indicator,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d,Risk_1d,Risk_7d,Risk_14d,Risk_30d,Risk_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0.0,21.2,40.7,60.2,65.5,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
1,1-you.njalla.no,28.1,78.0,127.9,196.6,156.0,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
4,1.234.83.26,53.9,104.7,117.0,137.1,117.0,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5,1.4.195.14,8.0,32.2,46.9,73.7,68.3,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
12,101.168.57.163,27.5,87.0,192.4,380.1,297.7,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk
...,...,...,...,...,...,...,...,...,...,...,...
5762,www.deepseek.com,12.0,31.5,36.5,41.0,37.0,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5763,www.deepseek.com.cdn.cloudflare.net,11.7,67.4,93.8,128.9,120.1,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5792,www.prosinthecity.com,12.5,33.4,66.7,137.6,141.8,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5795,www.sthda.com,16.8,68.9,90.3,107.1,97.9,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk


In [38]:
# Total TRRI scores across all (assumed active) indicators

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

# Turn into a clean DataFrame
total_trri_df = (
    pd.DataFrame(list(total_trri.items()), columns=["TRRI_Horizon", "Total_TRRI"])
    .sort_values("TRRI_Horizon")
    .reset_index(drop=True)
)

total_trri_df

,TRRI_Horizon,Total_TRRI
0,TRRI_14d,409462.9
1,TRRI_1d,199740.9
2,TRRI_30d,476080.5
3,TRRI_45d,420404.7
4,TRRI_7d,350395.5


# Save current TRRI totals to CSV
out_path = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI\TRRI_totals_2025-06-16.csv"  # adjust name/date as needed
total_trri_df.to_csv(out_path, index=False)
out_path

import os
import glob
import pandas as pd

trri_dir = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI"

# Get all TRRI total CSVs in the folder
csv_files = glob.glob(os.path.join(trri_dir, "TRRI_totals_*.csv"))
if len(csv_files) < 2:
    raise ValueError("Need at least two TRRI_totals_*.csv files to compare.")

# Sort by filename (assumes date is in the name and sortable, e.g. TRRI_totals_2025-06-16.csv)
csv_files_sorted = sorted(csv_files)

prev_path = csv_files_sorted[-2]  # previous file
curr_path = csv_files_sorted[-1]  # current file

prev_df = pd.read_csv(prev_path)
curr_df = pd.read_csv(curr_path)

compare = prev_df.merge(
    curr_df,
    on="TRRI_Horizon",
    suffixes=("_prev", "_curr")
)

compare["Pct_Change"] = (
    (compare["Total_TRRI_curr"] - compare["Total_TRRI_prev"])
    / compare["Total_TRRI_prev"]
) * 100

compare